# Ingestion & Chunking — the LlamaIndex way

**Notebook 03 · Phase 2 (Data preparation)** · Stack: `llama-index`

This is the **twin of Notebook 02**. Same sample documents, same eight sections, same
"golden" question — but implemented with **LlamaIndex** instead of LangChain. Read them
side by side to feel how the two frameworks differ on identical tasks.

The headline: LlamaIndex models chunking as **node parsers**, and the two most advanced
strategies — **sentence-window** and **hierarchical (small-to-big)** — are first-class
one-liners here, whereas Notebook 02 had to wire them by hand.

> Loading uses `OPENAI_API_KEY` from `../.env` for the embedding-based strategies (3–5),
> exactly like Notebooks 01–02.

## 0. Install dependencies

Run this cell **first**. It installs everything this notebook needs into the *active
kernel* using `%pip`, so it works in any fresh environment — if a package is already
present, pip simply skips it. **After the very first install, restart the kernel**
(Kernel → Restart Kernel) so newly installed packages are picked up, then run the rest
top to bottom.

> `numpy` is pinned `<2` because the OCR stack (opencv / rapidocr) needs it.

In [1]:
# Install dependencies into the ACTIVE kernel (idempotent — skips what's already there).
%pip install -q \
    "numpy<2" \
    llama-index llama-index-readers-file llama-index-embeddings-openai llama-index-llms-openai \
    pymupdf4llm rapidocr-onnxruntime docx2txt beautifulsoup4 lxml \
    tiktoken python-dotenv
print("✅ Dependencies ready. If this was a fresh install, restart the kernel now, then re-run.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Users/mohamednoordeenalaudeen/Documents/GenAI-2026/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ Dependencies ready. If this was a fresh install, restart the kernel now, then re-run.


## 1. Setup

`Settings` is LlamaIndex's global config: set the LLM and embedding model once and every
index/parser picks them up. We point both at OpenAI to match the other notebooks.

In [2]:
import warnings, os, re
warnings.filterwarnings("ignore")
import logging
# quiet per-request HTTP logs but keep LlamaIndex info (e.g. auto-merging messages)
for _n in ("httpx", "openai", "httpcore"):
    logging.getLogger(_n).setLevel(logging.WARNING)
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")
DATA = Path.cwd() / "data"

from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("LLM:", Settings.llm.model, "| embed:", Settings.embed_model.model_name)

OPENAI_API_KEY set: True
LLM: gpt-4o-mini | embed: text-embedding-3-small


### A helper to *see* nodes

LlamaIndex calls chunks **nodes**. `inspect_nodes()` mirrors the `inspect_chunks()` helper
from Notebook 02 so the two are directly comparable.

In [3]:
import tiktoken
_enc = tiktoken.get_encoding("cl100k_base")
def ntok(s): return len(_enc.encode(s))

def inspect_nodes(nodes, n_preview=3, width=200):
    texts = [n.get_content() for n in nodes]
    toks = [ntok(t) for t in texts] or [0]
    print(f"→ {len(texts)} nodes | tokens min/avg/max = {min(toks)}/{sum(toks)//len(toks)}/{max(toks)}")
    for i, t in enumerate(texts[:n_preview]):
        print(f"┃ [node {i}] {' '.join(t.split())[:width]}{'…' if len(t) > width else ''}")

## 2. Ingestion & parsing

Same lesson as Notebook 02: **most quality loss happens here**. LlamaIndex's signature
strength is loading — `SimpleDirectoryReader` ingests a whole folder of mixed formats in
one line and routes each file to the right reader.

### 2a. PDF — naive vs. layout-aware (the money demo)

`SimpleDirectoryReader`'s default PDF reader extracts raw text (table flattened). For
layout-aware Markdown we use `pymupdf4llm` and wrap the result in a `Document`.
*(The managed alternative is **LlamaParse** — best-in-class for tables/scanned PDFs — which
needs a `LLAMA_CLOUD_API_KEY`; we keep this notebook local and key-free.)*

In [4]:
from llama_index.core import SimpleDirectoryReader, Document
import pymupdf4llm

naive_docs = SimpleDirectoryReader(input_files=[str(DATA / "sample_report.pdf")]).load_data()
naive = "\n".join(d.text for d in naive_docs)

report_md = pymupdf4llm.to_markdown(str(DATA / "sample_report.pdf"))  # layout-aware

def table_region(t):
    i = t.find("Region"); return t[i:i + 320]

print("=== SimpleDirectoryReader default (naive) — table flattened ===")
print(table_region(naive))
print("\n=== pymupdf4llm (layout-aware) — table survives as Markdown ===")
print(table_region(report_md))

=== Document parser messages ===


Using RapidOCR and Tesseract for OCR processing.



=== SimpleDirectoryReader default (naive) — table flattened ===
Regional performance is broken out in the table below.
Region
Shipments
On-Time %
Revenue ($M)
Northwest
128,400
97.2%
14.6
West
204,910
98.1%
23.9
South
176,220
96.8%
19.4
Northeast
92,180
89.3%
11.2
Total
601,710
95.9%
69.1
Action items: (1) complete the Northeast warehouse migration by July 15; (2) reallocate two de

=== pymupdf4llm (layout-aware) — table survives as Markdown ===
Regional performance is broken out in the table below. 

|**Region**|**Shipments**|**On-Time %**|**Revenue ($M)**|
|---|---|---|---|
|Northwest|128,400|97.2%|14.6|
|West|204,910|98.1%|23.9|
|South|176,220|96.8%|19.4|
|Northeast|92,180|89.3%|11.2|
|**Total**|**601,710**|**95.9%**|**69.1**|



Action items: (1) complete 


Same result as the LangChain notebook: the naive reader loses the row/column structure;
the layout-aware parser keeps it. Parsing quality is framework-independent — it's about
the parser you choose, not the orchestration library.

### 2b. Load a whole folder in one line

This is where LlamaIndex shines: point `SimpleDirectoryReader` at the folder and it loads
PDF, HTML, and Word together, tagging each node with its source file.

In [5]:
docs = SimpleDirectoryReader(
    input_dir=str(DATA),
    required_exts=[".pdf", ".html", ".docx"],  # skip the .png (needs an image reader)
).load_data()

print(f"loaded {len(docs)} document(s) from the folder:")
for d in docs[:8]:
    print("  •", d.metadata.get("file_name"), "|", ntok(d.text), "tokens")

# grab the contract text for the chunking demos below
contract = next(d.text for d in docs if d.metadata.get("file_name", "").endswith(".docx"))

loaded 3 document(s) from the folder:
  • sample_contract.docx | 278 tokens
  • sample_page.html | 377 tokens
  • sample_report.pdf | 215 tokens


### 2c. Scanned image — OCR

LlamaIndex can wrap OCR output as a `Document`. We reuse the same framework-agnostic OCR
engine (RapidOCR) from Notebook 02 — OCR is a parsing concern, not a framework one.

In [6]:
from rapidocr_onnxruntime import RapidOCR
_ocr = RapidOCR()
result, _ = _ocr(str(DATA / "sample_scanned.png"))
ocr_text = "\n".join(line[1] for line in result) if result else "(no text)"
ocr_doc = Document(text=ocr_text, metadata={"source": "sample_scanned.png", "via": "OCR"})
print("=== OCR → LlamaIndex Document ===")
print(ocr_doc.text)

=== OCR → LlamaIndex Document ===
INTERNALMEMO
TO：
All warehouse staff
FRoM: Operations, Northwind Logistics
DATE: 2024-06-14
RE：
Incident INC-5521
The claims-search service had a 22-minute outage on
2024-06-14 causedby a Qdrant snapshot restorethat
exhausted disk.Fix: volume size increased and a disk-
usage alert added at 80%. Owner: platform team.


**Takeaway for Part 2:** identical to Notebook 02 — parsing sets the ceiling. What differs
is ergonomics: `SimpleDirectoryReader` gives you folder-level, multi-format loading (and via
LlamaHub, hundreds of connectors) in a single call.

## 3. Chunking — six strategies as LlamaIndex node parsers

A **node** is LlamaIndex's chunk. Every strategy below is a *node parser* that turns
`Document`s into `Node`s. We use the same sources as Notebook 02: the layout-aware report
Markdown and the contract text.

In [7]:
print("report_md tokens:", ntok(report_md), "| contract tokens:", ntok(contract))

report_md tokens: 254 | contract tokens: 278


### Strategy 1 — Fixed-size / token-based  *(baseline, naive)*

**LlamaIndex API:** `TokenTextSplitter`

In [8]:
from llama_index.core.node_parser import TokenTextSplitter
s1 = TokenTextSplitter(chunk_size=80, chunk_overlap=10)
inspect_nodes(s1.get_nodes_from_documents([Document(text=report_md)]))

→ 4 nodes | tokens min/avg/max = 46/65/78
┃ [node 0] # **Northwind Logistics — Q2 2024 Operations Report** This report summarizes regional shipment performance for the second quarter of 2024. Overall shipment volume grew 8.4% quarter over quarter, drive…
┃ [node 1] migration temporarily reduced throughput. Regional performance is broken out in the table below. |**Region**|**Shipments**|**On-Time %**|**Revenue ($M)**| |---|---|---|---| |Northwest|128,400|97.2%|14…
┃ [node 2] |West|204,910|98.1%|23.9| |South|176,220|96.8%|19.4| |Northeast|92,180|89.3%|11.2| |**Total**|**601,710**|**95.9%**|**69.1**|


### Strategy 2 — Sentence-aware splitting  *(sane default)*

LlamaIndex's default. `SentenceSplitter` keeps sentences whole while targeting a token
window — the counterpart to LangChain's `RecursiveCharacterTextSplitter`. **Careful:**
its `chunk_size` counts *tokens*, while LangChain's default counts *characters* — the same
number is not the same budget across the two frameworks.

**LlamaIndex API:** `SentenceSplitter`

In [9]:
from llama_index.core.node_parser import SentenceSplitter
# NOTE: SentenceSplitter's chunk_size counts TOKENS (report_md is 254 tokens total), while
# LangChain's RecursiveCharacterTextSplitter in Notebook 02 counts CHARACTERS — the same
# number means something different in each framework. chunk_size=400 tokens would exceed
# the whole document and produce a single node, so we use 100 tokens here to actually show
# the sentence-aware split happening.
s2 = SentenceSplitter(chunk_size=100, chunk_overlap=20)
inspect_nodes(s2.get_nodes_from_documents([Document(text=report_md)]))

→ 3 nodes | tokens min/avg/max = 84/93/99
┃ [node 0] # **Northwind Logistics — Q2 2024 Operations Report** This report summarizes regional shipment performance for the second quarter of 2024. Overall shipment volume grew 8.4% quarter over quarter, drive…
┃ [node 1] Regional performance is broken out in the table below. |**Region**|**Shipments**|**On-Time %**|**Revenue ($M)**| |---|---|---|---| |Northwest|128,400|97.2%|14.6| |West|204,910|98.1%|23.9| |South|176,2…
┃ [node 2] 220|96.8%|19.4| |Northeast|92,180|89.3%|11.2| |**Total**|**601,710**|**95.9%**|**69.1**| Action items: (1) complete the Northeast warehouse migration by July 15; (2) reallocate two delivery routes fro…


### Strategy 3 — Semantic chunking  *(splits at meaning boundaries)*

Cuts where embedding similarity drops between adjacent sentences.

**LlamaIndex API:** `SemanticSplitterNodeParser` (+ an embed model)

In [10]:
from llama_index.core.node_parser import SemanticSplitterNodeParser
s3 = SemanticSplitterNodeParser(
    embed_model=Settings.embed_model, buffer_size=1, breakpoint_percentile_threshold=95,
)
inspect_nodes(s3.get_nodes_from_documents([Document(text=contract)]))

→ 2 nodes | tokens min/avg/max = 31/139/247
┃ [node 0] Master Services Agreement This Master Services Agreement ("Agreement") is entered into between Northwind Logistics, Inc. ("Provider") and the Client, effective as of the date of last signature. 1. Sco…
┃ [node 1] 6. Limitation of Liability Neither party's aggregate liability shall exceed the fees paid under this Agreement in the twelve (12) months preceding the claim.


### Strategy 4 — Sentence-window retrieval  *(retrieve small, expand at generation)*

**This is a LlamaIndex one-liner.** `SentenceWindowNodeParser` indexes each sentence but
stores its neighbor window in metadata; at query time `MetadataReplacementPostProcessor`
swaps the matched sentence for its window. Compare this to the manual implementation in
Notebook 02 — same idea, far less code.

**LlamaIndex API:** `SentenceWindowNodeParser` + `MetadataReplacementPostProcessor`

In [11]:
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceWindowNodeParser

sw_parser = SentenceWindowNodeParser.from_defaults(
    window_size=2, window_metadata_key="window", original_text_metadata_key="original_text",
)
sw_nodes = sw_parser.get_nodes_from_documents([Document(text=contract)])
sw_index = VectorStoreIndex(sw_nodes)

query = "What is the home-internet stipend for remote employees?"
hit = sw_index.as_retriever(similarity_top_k=1).retrieve(query)[0]
print("QUERY:", query)
print("\n-- Matched SENTENCE (original_text) --\n", hit.node.metadata["original_text"])
print("\n-- Expanded WINDOW (what the LLM gets) --\n", hit.node.metadata["window"])

QUERY: What is the home-internet stipend for remote employees?

-- Matched SENTENCE (original_text) --
 Remote-Work Reimbursement (HR-2024-17 §7.3)

Employees working remotely at least three (3) days per week are eligible for a home-internet stipend of $45 per month, reimbursed via the monthly expense report. 

-- Expanded WINDOW (what the LLM gets) --
 Provider may suspend services immediately for non-payment exceeding sixty (60) days.

 5.  Remote-Work Reimbursement (HR-2024-17 §7.3)

Employees working remotely at least three (3) days per week are eligible for a home-internet stipend of $45 per month, reimbursed via the monthly expense report.  Contractors are not eligible.

 6. 


### Strategy 5 — Hierarchical / small-to-big  *(auto-merging)*

Also first-class here. `HierarchicalNodeParser` builds parent/child nodes at multiple sizes;
`AutoMergingRetriever` matches small leaves but, when enough leaves of the same parent are
hit, **automatically merges them up to the parent** and returns that. Watch the log say
"Merging N nodes into parent node."

**LlamaIndex API:** `HierarchicalNodeParser` + `get_leaf_nodes` + `AutoMergingRetriever`

In [12]:
from llama_index.core import StorageContext
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.storage.docstore import SimpleDocumentStore

h_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[400, 100])
all_nodes = h_parser.get_nodes_from_documents([Document(text=contract)])
leaves = get_leaf_nodes(all_nodes)

docstore = SimpleDocumentStore(); docstore.add_documents(all_nodes)
sc = StorageContext.from_defaults(docstore=docstore)
h_index = VectorStoreIndex(leaves, storage_context=sc)

amr = AutoMergingRetriever(h_index.as_retriever(similarity_top_k=6), sc, verbose=True)
merged = amr.retrieve("remote work home internet stipend eligibility contractors")
print(f"\ntotal nodes: {len(all_nodes)} | leaves: {len(leaves)} | returned after merge: {len(merged)}")
print("\n-- Returned node (merged parent, full context) --\n", merged[0].node.get_content()[:400], "…")

2026-07-10 18:46:38,319 - INFO - > Merging 5 nodes into parent node.
> Parent node id: 382701a1-1728-4446-b980-f8de8bcb23a1.
> Parent node text: Master Services Agreement

This Master Services Agreement ("Agreement") is entered into between N...



> Merging 5 nodes into parent node.
> Parent node id: 382701a1-1728-4446-b980-f8de8bcb23a1.
> Parent node text: Master Services Agreement

This Master Services Agreement ("Agreement") is entered into between N...


total nodes: 6 | leaves: 5 | returned after merge: 1

-- Returned node (merged parent, full context) --
 Master Services Agreement

This Master Services Agreement ("Agreement") is entered into between Northwind Logistics, Inc. ("Provider") and the Client, effective as of the date of last signature.

1. Scope of Services

Provider shall deliver freight logistics and warehousing services as described in each executed Statement of Work.

2. Term

This Agreement remains in effect for an initial term of t …


### Strategy 6 — Document-type-aware chunking

Structure-aware parsers: `MarkdownNodeParser` splits on headers, `HTMLNodeParser` on HTML
tags. *(LlamaIndex also ships `CodeSplitter` for source code via tree-sitter — the LangChain
notebook covers code splitting with `from_language`.)*

**LlamaIndex APIs:** `MarkdownNodeParser`, `HTMLNodeParser`

In [13]:
from llama_index.core.node_parser import MarkdownNodeParser, HTMLNodeParser

markdown_doc = """# Support FAQ
## Shipping
Standard shipping takes 3-5 business days.
## Returns
Returns are accepted within 30 days of delivery.
## Tracking
Every shipment includes a tracking number."""
print("MARKDOWN nodes (split on headers):")
inspect_nodes(MarkdownNodeParser().get_nodes_from_documents([Document(text=markdown_doc)]),
              n_preview=3, width=90)

html_raw = (DATA / "sample_page.html").read_text(encoding="utf-8")
print("\nHTML nodes (split on tags):")
inspect_nodes(HTMLNodeParser(tags=["h1", "h2", "p", "ul", "table"]).get_nodes_from_documents(
    [Document(text=html_raw)]), n_preview=3, width=90)

MARKDOWN nodes (split on headers):
→ 4 nodes | tokens min/avg/max = 3/9/13
┃ [node 0] # Support FAQ
┃ [node 1] ## Shipping Standard shipping takes 3-5 business days.
┃ [node 2] ## Returns Returns are accepted within 30 days of delivery.

HTML nodes (split on tags):
→ 10 nodes | tokens min/avg/max = 1/16/32
┃ [node 0] Northwind Logistics — Support FAQ
┃ [node 1] Answers to common questions about shipping, tracking, and returns.
┃ [node 2] Shipping


## 4. Summary — LlamaIndex vs. LangChain on the same task

| # | Strategy | LlamaIndex | LangChain (Notebook 02) |
|---|----------|------------|--------------------------|
| 1 | Fixed / token | `TokenTextSplitter` | `TokenTextSplitter` |
| 2 | Sentence / recursive | `SentenceSplitter` | `RecursiveCharacterTextSplitter` |
| 3 | Semantic | `SemanticSplitterNodeParser` | `SemanticChunker` (experimental) |
| 4 | **Sentence-window** | `SentenceWindowNodeParser` + postprocessor ✅ *one-liner* | *manual metadata window* |
| 5 | **Small-to-big** | `HierarchicalNodeParser` + `AutoMergingRetriever` ✅ *built-in merge* | `ParentDocumentRetriever` |
| 6 | Doc-type-aware | `MarkdownNodeParser`, `HTMLNodeParser`, `CodeSplitter` | `MarkdownHeaderTextSplitter`, `HTMLHeaderTextSplitter`, `from_language` |

### What the comparison shows

- **Loading:** LlamaIndex's `SimpleDirectoryReader` (+ LlamaHub) is the smoother multi-source
  loader; LangChain uses one loader class per format.
- **Advanced chunking:** sentence-window and small-to-big are *native* in LlamaIndex
  (strategies 4–5), while LangChain needed manual wiring — this is LlamaIndex's real edge for
  retrieval-aware chunking.
- **Parsing quality is identical** across both — it depends on the parser (pymupdf4llm,
  LlamaParse, Unstructured), not the orchestration framework.
- **Rule of thumb:** LangChain for flexible orchestration/generation (Notebook 01), LlamaIndex
  for data ingestion + sophisticated chunking/retrieval. Many production stacks use both.

### A 7th technique worth knowing: Contextual Retrieval

Every strategy above decides *where* to cut a document. A complementary technique
decides *what to prepend* to each node before it's embedded: Anthropic's **Contextual
Retrieval** (2024) uses an LLM to generate a short chunk-level summary of how that
chunk fits the whole document, and prepends it before embedding and before BM25
indexing — reported at ~35–67% fewer failed retrievals depending on what it's
combined with. It's framework-agnostic (works identically with LlamaIndex node
parsers or LangChain splitters) and composes with every strategy above rather than
replacing any of them — one extra LLM call per chunk at indexing time, not per query.

### Next
With clean parsing and the right chunking in hand, the next step is **embeddings + a vector
database + retrieval/reranking** — turning these nodes into a searchable index.